# 02. Data Cleansing, Imputation and Relational Normalization
This notebook executes domain-specific cleansing, holiday forward-filling, calendar reindexing, and boundary assertions.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

raw_dir = Path('../data/raw')
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)


## NAV Holiday Reindexing and Forward-Fill Imputation
Aligning daily NAV series to a continuous calendar range and forward-filling market holidays.


In [2]:
df_nav = pd.read_csv(raw_dir / '02_nav_history.csv')
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav = df_nav.drop_duplicates(subset=['amfi_code', 'date']).sort_values(['amfi_code', 'date'])

reindexed = []
for amfi, group in df_nav.groupby('amfi_code'):
    cal_idx = pd.date_range(group['date'].min(), group['date'].max(), freq='D', name='date')
    g = group.set_index('date').reindex(cal_idx)
    g['amfi_code'] = amfi
    g['nav'] = g['nav'].ffill().bfill()
    g = g.reset_index()
    g['daily_return_pct'] = g['nav'].pct_change() * 100.0
    reindexed.append(g)

df_nav_clean = pd.concat(reindexed, ignore_index=True)
assert (df_nav_clean['nav'] > 0).all()
df_nav_clean.rename(columns={'date': 'nav_date'}, inplace=True)
df_nav_clean.to_csv(processed_dir / 'fact_nav.csv', index=False)
print('NAV calendar reindexing complete. Total records:', len(df_nav_clean))


NAV calendar reindexing complete. Total records: 64320


## Transaction Standardization and Validation
Validating positive amounts, standardizing transaction types, and verifying KYC status enums.


In [3]:
df_txn = pd.read_csv(raw_dir / '08_investor_transactions.csv')
df_txn['transaction_date'] = pd.to_datetime(df_txn['transaction_date'])
df_txn['amount_inr'] = pd.to_numeric(df_txn['amount_inr'])
assert (df_txn['amount_inr'] > 0).all()

df_txn['transaction_type'] = df_txn['transaction_type'].str.strip()
assert set(df_txn['transaction_type'].unique()).issubset({'SIP', 'Lumpsum', 'Redemption'})
assert set(df_txn['kyc_status'].dropna().unique()).issubset({'Verified', 'Pending'})

df_txn.to_csv(processed_dir / 'fact_transactions.csv', index=False)
print('Transaction normalization verified. Total transaction records:', len(df_txn))


Transaction normalization verified. Total transaction records: 32778


## Scheme Performance Boundary Assertions
Enforcing regulatory bounds on expense ratios and verifying return numeric dtypes.


In [4]:
df_perf = pd.read_csv(raw_dir / '07_scheme_performance.csv')
df_perf['expense_ratio_pct'] = df_perf['expense_ratio_pct'].astype(float)
assert ((df_perf['expense_ratio_pct'] >= 0.1) & (df_perf['expense_ratio_pct'] <= 2.5)).all()

for col in ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct']:
    df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')

df_perf.to_csv(processed_dir / 'fact_performance.csv', index=False)
print('Scheme performance verification passed. Expense ratios strictly bounded between 0.1% and 2.5%.')


Scheme performance verification passed. Expense ratios strictly bounded between 0.1% and 2.5%.
